# imports

In [1]:
import os
import random
import numpy as np
import torch
import torchaudio
import librosa
from pathlib import Path
from datasets import load_from_disk, Dataset, DatasetDict, Audio
from typing import Optional
from tqdm.auto import tqdm


# Load Data

In [2]:
PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / '.git').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_PATH       = PROJECT_ROOT / 'data' / 'synthetic' / 'v3'
SYNTHETIC_AUDIO_DIR = PROJECT_ROOT / 'data' / 'synthetic' / 'audio'
TEANGLANN_AUDIO_DIR = PROJECT_ROOT / 'data' / 'teanglann' / 'wav_files'
OUTPUT_PATH        = PROJECT_ROOT / 'data' / 'synthetic' / 'l2_synth_wavenet_train'


In [3]:
paired_ds = load_from_disk(str(DATASET_PATH))
print(paired_ds)

Parameter 'format_kwargs'={} of the transform datasets.arrow_dataset.Dataset.set_format couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Dataset({
    features: ['audio', 'phonetic', 'English ASR transcriptions', 'synthetic_audio_path'],
    num_rows: 19136
})


# Inspect Data

In [4]:
paired_ds[0]

{'audio': {'path': None,
  'array': array([0., 0., 0., ..., 0., 0., 0.], shape=(13440,)),
  'sampling_rate': 16000},
 'phonetic': 'k a ɾ ə b ə d',
 'English ASR transcriptions': 'kʌɹəbʌd',
 'synthetic_audio_path': '/home/peter/Desktop/thesis/ThesisProject/data/synthetic/audio/carbad.mp3'}

Not great that I am saving audio inconsistently, remember not to do this in the future.

# Prepare Data

First let's load the audio into the dataset so the audio is presented in the same way

In [5]:
from datasets import Audio

paired_ds = paired_ds.cast_column("synthetic_audio_path", Audio(sampling_rate=16000))
paired_ds = paired_ds.rename_column("synthetic_audio_path", "synthetic_audio")

In [6]:
paired_ds[0]

{'audio': {'path': None,
  'array': array([0., 0., 0., ..., 0., 0., 0.], shape=(13440,)),
  'sampling_rate': 16000},
 'phonetic': 'k a ɾ ə b ə d',
 'English ASR transcriptions': 'kʌɹəbʌd',
 'synthetic_audio': {'path': '/home/peter/Desktop/thesis/ThesisProject/data/synthetic/audio/carbad.mp3',
  'array': array([ 7.99360578e-15, -8.88178420e-15,  9.76996262e-15, ...,
          1.27620979e-07, -3.58589034e-07, -3.13843884e-09], shape=(15360,)),
  'sampling_rate': 16000}}

forgot to normalize transcriptions to same wav2vec2-friendly format as the teanglann scrapings. better for spaces between characters to keep diacritics together with their phones

In [7]:
import json


SYNTH_VOCAB = {' ': 0, 'aɪ': 1, 'aʊ': 2, 'b': 3, 'd': 4, 'eɪ': 5, 'f': 6, 'g': 7,
               'h': 8, 'iː': 9, 'j': 10, 'k': 11, 'l': 12, 'l̩': 13, 'm': 14, 'm̩': 15,
               'n': 16, 'n̩': 17, 'oʊ': 18, 'p': 19, 's': 20, 't': 21, 'uː': 22, 'v': 23,
               'w': 24, 'z': 25, 'æ': 26, 'ð': 27, 'ŋ': 28, 'ŋ̍': 29, 'ɑː': 30, 'ɔː': 31,
               'ɔɪ': 32, 'ə': 33, 'ɚ': 35, 'ɛ': 36,
               'ɪ': 40, 'ɹ': 41, 'ʃ': 44, 'ʊ': 46, 'ʌ': 47,
               'ʒ': 48, 'ʤ': 50, 'ʧ': 51, 'θ': 52}

# Partition into 2-codepoint and 1-codepoint sets for greedy matching
_PHONES_2 = {k for k in SYNTH_VOCAB if len(k) == 2 and k != ' '}
_PHONES_1 = {k for k in SYNTH_VOCAB if len(k) == 1 and k != ' '}


def _space_separate(ipa: str) -> str:
    """Insert spaces between phones in a collapsed synthetic IPA string."""
    if ' ' in ipa:          # already separated
        return ipa
    phones, i = [], 0
    while i < len(ipa):
        if ipa[i:i+2] in _PHONES_2:
            phones.append(ipa[i:i+2])
            i += 2
        else:
            if ipa[i] in _PHONES_1:
                phones.append(ipa[i])
            i += 1
    return ' '.join(phones)

In [8]:
unknown = set()
for item in paired_ds:
    for phone in _space_separate(item['English ASR transcriptions']).split():
        if phone not in SYNTH_VOCAB:
            unknown.add(phone)

print(unknown if unknown else "All phones covered")


All phones covered


In [9]:
paired_ds = paired_ds.map(
    lambda item: {'English ASR transcriptions': _space_separate(item['English ASR transcriptions'])},
    desc='Normalising synthetic IPA',
)
print('Before:', 'kʌɾəbʌd')
print('After: ', _space_separate('kʌɾəbʌd'))

Normalising synthetic IPA:   0%|          | 0/19136 [00:00<?, ? examples/s]

Before: kʌɾəbʌd
After:  k ʌ ə b ʌ d


In [10]:
paired_ds[0]

{'audio': {'path': None,
  'array': array([0., 0., 0., ..., 0., 0., 0.], shape=(13440,)),
  'sampling_rate': 16000},
 'phonetic': 'k a ɾ ə b ə d',
 'English ASR transcriptions': 'k ʌ ɹ ə b ʌ d',
 'synthetic_audio': {'path': '/home/peter/Desktop/thesis/ThesisProject/data/synthetic/audio/carbad.mp3',
  'array': array([ 7.99360578e-15, -8.88178420e-15,  9.76996262e-15, ...,
          1.27620979e-07, -3.58589034e-07, -3.13843884e-09], shape=(15360,)),
  'sampling_rate': 16000}}

## strip diacritics

In [11]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from scripts.data_handling.collapse_phonemes import collapse_phones

In [12]:
def normalize_row(row):
    row['phonetic'] = "".join(collapse_phones(row['phonetic']))
    row['English ASR transcriptions'] = "".join(collapse_phones(row['English ASR transcriptions']))
    return row

In [13]:
paired_ds = paired_ds.map(
    lambda row: normalize_row(row),
    load_from_cache_file=False,
    desc="Normalizing phonetic columns"
)

Normalizing phonetic columns:   0%|          | 0/19136 [00:00<?, ? examples/s]

In [14]:
paired_ds[0]

{'audio': {'path': None,
  'array': array([0., 0., 0., ..., 0., 0., 0.], shape=(13440,)),
  'sampling_rate': 16000},
 'phonetic': 'k a ɾ ə b ə d',
 'English ASR transcriptions': 'k ʌ ɹ ə b ʌ d',
 'synthetic_audio': {'path': '/home/peter/Desktop/thesis/ThesisProject/data/synthetic/audio/carbad.mp3',
  'array': array([ 7.99360578e-15, -8.88178420e-15,  9.76996262e-15, ...,
          1.27620979e-07, -3.58589034e-07, -3.13843884e-09], shape=(15360,)),
  'sampling_rate': 16000}}

In [15]:
len(paired_ds[0]['phonetic'])

13

## make paired set

In [16]:
TARGET_SR = 16_000
IPA_SEPARATOR = " "
RANDOM_SEED = 42


def _to_mono_16k(audio_array: np.ndarray, source_sr: int) -> np.ndarray:
    """Resample and convert to mono float32 at TARGET_SR."""
    waveform = torch.from_numpy(audio_array.astype(np.float32))
    if waveform.ndim == 1:
        waveform = waveform.unsqueeze(0)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if source_sr != TARGET_SR:
        resampler = torchaudio.transforms.Resample(orig_freq=source_sr, new_freq=TARGET_SR)
        waveform = resampler(waveform)
    return waveform.squeeze(0).numpy()


def _make_l2_pair(item: dict, idx: int) -> dict:
    """Map function: concatenate native + TTS audio for one word."""
    ref_audio = _to_mono_16k(item["audio"]["array"], item["audio"]["sampling_rate"])

    synth_array = item["synthetic_audio"]["array"]
    synth_sr = item["synthetic_audio"]["sampling_rate"]
    synth_audio = _to_mono_16k(synth_array, synth_sr)

    # Per-item seeding gives the same reproducibility as a shared rng.
    if random.Random(RANDOM_SEED + idx).random() < 0.5: # Vary order so ordering isn't introducing bias
        combined = np.concatenate([ref_audio, synth_audio])
        ipa = item["phonetic"] + IPA_SEPARATOR + item["English ASR transcriptions"]
        order = "ref_first"
    else:
        combined = np.concatenate([synth_audio, ref_audio])
        ipa = item["English ASR transcriptions"] + IPA_SEPARATOR + item["phonetic"]
        order = "synth_first"

    return {"audio": {"array": combined, "sampling_rate": TARGET_SR}, "ipa": ipa, "order": order}


def build_l2_training_set(dataset: Dataset, seed: Optional[int] = RANDOM_SEED) -> Dataset:
    """
    Build an adversarially-concatenated dataset directly from the paired dataset.

    Input columns:  audio, phonetic, synthetic_audio, English ASR transcriptions
    Output columns: audio, ipa, order
    """
    dataset = dataset.cast_column("audio", Audio(sampling_rate=TARGET_SR))
    out_ds = dataset.map(
        _make_l2_pair,
        with_indices=True,
        remove_columns=["phonetic", "synthetic_audio", "English ASR transcriptions"],
        desc="Building L2 training set",
    )
    return out_ds.cast_column("audio", Audio(sampling_rate=TARGET_SR))


In [17]:
l2_ds = build_l2_training_set(paired_ds)

Building L2 training set:   0%|          | 0/19136 [00:00<?, ? examples/s]

In [18]:
print(l2_ds)
print('\nSample IPA:', l2_ds[0]['ipa'])

Dataset({
    features: ['audio', 'ipa', 'order'],
    num_rows: 19136
})

Sample IPA: k ʌ ɹ ə b ʌ d k a ɾ ə b ə d


# Split dataset
.8/.1/.1 split to be comparable to other models

In [19]:
l2_ds

Dataset({
    features: ['audio', 'ipa', 'order'],
    num_rows: 19136
})

In [20]:
ds_temp = l2_ds.train_test_split(test_size=0.2, seed=RANDOM_SEED)
ds_final = ds_temp["test"].train_test_split(test_size=0.5, seed=RANDOM_SEED)

final_ds = DatasetDict({
    "train":      ds_temp["train"],
    "validation": ds_final["train"],
    "test":       ds_final["test"],
})

In [21]:
final_ds

DatasetDict({
    train: Dataset({
        features: ['audio', 'ipa', 'order'],
        num_rows: 15308
    })
    validation: Dataset({
        features: ['audio', 'ipa', 'order'],
        num_rows: 1914
    })
    test: Dataset({
        features: ['audio', 'ipa', 'order'],
        num_rows: 1914
    })
})

# Save datasets

In [22]:
print(OUTPUT_PATH)

/home/peter/Desktop/thesis/ThesisProject/data/synthetic/l2_synth_wavenet_train


In [23]:
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
final_ds.save_to_disk(str(OUTPUT_PATH))
print(f'Saved to {OUTPUT_PATH}')


Saving the dataset (0/2 shards):   0%|          | 0/15308 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1914 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1914 [00:00<?, ? examples/s]

Saved to /home/peter/Desktop/thesis/ThesisProject/data/synthetic/l2_synth_wavenet_train


# Build IPA vocab

In [24]:
train_phonetics = [phone for x in final_ds["train"] for phone in x['ipa'].split()]
valid_phonetics = [phone for x in final_ds["validation"] for phone in x['ipa'].split()]
test_phonetics = [phone for x in final_ds["test"] for phone in x['ipa'].split()]

print("num of train phones:\t", len(set(train_phonetics)))
print("num of valid phones:\t", len(set(valid_phonetics)))
print("num of test phones:\t", len(set(test_phonetics)))

num of train phones:	 75
num of valid phones:	 74
num of test phones:	 74


In [25]:
vocab_train = list(set(train_phonetics)) + [' ']
vocab_valid = list(set(valid_phonetics)) + [' ']
vocab_test  = list(set(test_phonetics)) + [' ']

In [26]:
vocab_list = list(set(vocab_train + vocab_valid + vocab_test))
vocab_dict = {v: k for k, v in enumerate(sorted(vocab_list))}

print(vocab_dict)

{' ': 0, 'a': 1, 'ai': 2, 'au': 3, 'aɪ': 4, 'aʊ': 5, 'aː': 6, 'b': 7, 'bʲ': 8, 'c': 9, 'd': 10, 'dʲ': 11, 'e': 12, 'eɪ': 13, 'eː': 14, 'f': 15, 'fʲ': 16, 'g': 17, 'h': 18, 'hʲ': 19, 'i': 20, 'ia': 21, 'iː': 22, 'iˑə': 23, 'j': 24, 'k': 25, 'l': 26, 'lʲ': 27, 'm': 28, 'mʲ': 29, 'n': 30, 'nʲ': 31, 'o': 32, 'oʊ': 33, 'oː': 34, 'p': 35, 'pʲ': 36, 's': 37, 't': 38, 'tʲ': 39, 'u': 40, 'ua': 41, 'uː': 42, 'uˑə': 43, 'v': 44, 'vʲ': 45, 'w': 46, 'x': 47, 'z': 48, 'zʲ': 49, 'æ': 50, 'ç': 51, 'ð': 52, 'ŋ': 53, 'ɑː': 54, 'ɒ': 55, 'ɔɪ': 56, 'ɔː': 57, 'ə': 58, 'ɚ': 59, 'ɛ': 60, 'ɟ': 61, 'ɡ': 62, 'ɣ': 63, 'ɪ': 64, 'ɲ': 65, 'ɹ': 66, 'ɾ': 67, 'ɾʲ': 68, 'ʃ': 69, 'ʊ': 70, 'ʌ': 71, 'ʒ': 72, 'ʤ': 73, 'ʧ': 74, 'θ': 75}


In [27]:
# make the space more intuitive to understand
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]

vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)
len(vocab_dict)

78

In [28]:
print(vocab_dict)

{'a': 1, 'ai': 2, 'au': 3, 'aɪ': 4, 'aʊ': 5, 'aː': 6, 'b': 7, 'bʲ': 8, 'c': 9, 'd': 10, 'dʲ': 11, 'e': 12, 'eɪ': 13, 'eː': 14, 'f': 15, 'fʲ': 16, 'g': 17, 'h': 18, 'hʲ': 19, 'i': 20, 'ia': 21, 'iː': 22, 'iˑə': 23, 'j': 24, 'k': 25, 'l': 26, 'lʲ': 27, 'm': 28, 'mʲ': 29, 'n': 30, 'nʲ': 31, 'o': 32, 'oʊ': 33, 'oː': 34, 'p': 35, 'pʲ': 36, 's': 37, 't': 38, 'tʲ': 39, 'u': 40, 'ua': 41, 'uː': 42, 'uˑə': 43, 'v': 44, 'vʲ': 45, 'w': 46, 'x': 47, 'z': 48, 'zʲ': 49, 'æ': 50, 'ç': 51, 'ð': 52, 'ŋ': 53, 'ɑː': 54, 'ɒ': 55, 'ɔɪ': 56, 'ɔː': 57, 'ə': 58, 'ɚ': 59, 'ɛ': 60, 'ɟ': 61, 'ɡ': 62, 'ɣ': 63, 'ɪ': 64, 'ɲ': 65, 'ɹ': 66, 'ɾ': 67, 'ɾʲ': 68, 'ʃ': 69, 'ʊ': 70, 'ʌ': 71, 'ʒ': 72, 'ʤ': 73, 'ʧ': 74, 'θ': 75, '|': 0, '[UNK]': 76, '[PAD]': 77}


In [30]:

# save vocab.json
import json
with open(str(OUTPUT_PATH)+"/vocab.json", 'w') as vocab_file:
    json.dump(vocab_dict, vocab_file)